# Software di un negozio di prodotti vegani
Questo progetto consiste nel realizzare un software per la gestione di un negozio di prodotti vegani.
Il software ha le seguenti funzionalità:
- Registrare nuovi prodotti, con nome, quantità, prezzo di vendita e prezzo di acquisto.
- Elencare tutti i prodotti presenti.
- Registrare le vendite effettuate.
- Mostrare i profitti lordi e netti.
- Mostrare un menu di aiuto con tutti i comandi disponibili.

Il software è testuale, quindi utilizzabile da riga di comando.

In [ ]:
"""
--------------------------- README ---------------------------
This software takes input in Italian and returns output in Italian.
For example: file names, exceptions, commands.
"""

#-------------------------------------------------------------------------------

import json # Data la struttura gerarchica e annidata dei dati, flessibilità
import os

#-------------------------------------------------------------------------------
# File per inventario e vendite. Vengono salvati nella directory di lavoro
inventory_file="inventario.json"
sales_file="vendite.json"

#-------------------------------------------------------------------------------

def load_inventory():
  """
  Questa funzione carica l'inventario dal file.
  Se il file non esiste restituisce una lista vuota. Se ci sono problemi
  nel caricamento restituisce una eccezione con messaggio e una lista vuota.
  """
  if os.path.exists(inventory_file):
    try:
      with open(inventory_file, "r", encoding="utf-8") as f:
        return json.load(f)
    except (json.JSONDecodeError, IOError):
      print("Errore nel caricamento file JSON per l'inventario.")
      return []
  else:
    return []

#-------------------------------------------------------------------------------

def save_inventory(inventory):
  """
  Questa funzione salva l'inventario nel file (indentazione=4 spazi).
  """
  with open(inventory_file, "w", encoding="utf-8") as f:
    json.dump(inventory, f, indent=4)

#-------------------------------------------------------------------------------

def load_sales():
  """
  Questa funzione carica le vendite dal file.
  Se il file non esiste restituisce una lista vuota. Se ci sono problemi
  nel caricamento restituisce una eccezione con messaggio e una lista vuota.
  """
  if os.path.exists(sales_file):
    try:
      with open(sales_file, "r", encoding="utf-8") as f:
        return json.load(f)
    except (json.JSONDecodeError, IOError):
      print("Errore nel caricamento file JSON per le vendite.")
      return []
  else:
    return []

#-------------------------------------------------------------------------------

def save_sales(sales):
  """
  Questa funzione salva le vendite nel file (indentazione=4 spazi).
  """
  with open(sales_file, "w", encoding="utf-8") as f:
    json.dump(sales, f, indent=4)

#-------------------------------------------------------------------------------

def search_product_by_name(inventory, name):
  """
  Questa funzione cerca un prodotto nell'inventario per nome (senza distinzione
  tra maiuscole e minuscole) e se il prodotto è nell'inventario restituisce
  il dizionario altrimenti None.
  """
  for product in inventory:
    if product['nome'].lower()==name.lower(): # conversione stringa in minuscolo
      return product
  return None

#-------------------------------------------------------------------------------

def add_product(inventory):
  """
  Questa funzione aggiunge un prodotto all'inventario e se il prodotto esiste
  già allora prende il valore attuale e ne incrementa la quantità con
  il nuovo record.
  """
  name=input("Nome del prodotto: ").strip()  # rimuove eventuali spazi vuoti digitati per errore

  # Gestione dell'input numerico per quantità, prezzo di acquisto e vendita
  try:
    quantity=int(input("Quantità: "))
  except ValueError:
    print("Errore: la quantità deve essere un numero intero.")
    return

  # Verifica che la quantità non sia negativa
  if quantity < 0:
    print("Errore: la quantità non può essere negativa.")
    return

  # Gestione nuovo record e prodotto già esistente (aggiornamento della quantità)
  product=search_product_by_name(inventory, name)
  if product:
    product['quantità']+=quantity
    print(f"AGGIUNTO: {quantity} X {product['nome']}")
  else:
    try:
      purchase_price=float(input("Prezzo di acquisto: "))
      sale_price=float(input("Prezzo di vendita: "))
    except ValueError:
      print("Errore: il prezzo deve essere un numero decimale con il punto come separatore.")
      return

    # Verifica che il prezzo non sia negativo
    if purchase_price < 0 or sale_price < 0:
      print("Errore: i prezzi non possono essere negativi.")
      return

    new_product={
      "nome": name,
      "quantità": quantity,
      "prezzo_acquisto": purchase_price,
      "prezzo_vendita": sale_price
    }
    inventory.append(new_product)
    print(f"AGGIUNTO: {quantity} X {name}")
  save_inventory(inventory)

#-------------------------------------------------------------------------------

def list_products(inventory):
  """
  Questa funzione restituisce un elenco di tutti i prodotti (con valùta €)
  presenti nell'inventario e con quantità maggiore di 0.
  Se non ci sono prodotti nel magazzino stampa un messaggio appropriato.
  """
  #      È implementato un filtro per mostrare solo prodotti in magazzino con
  #      quantità>0. Se si vogliono vedere tutti i prodotti (anche con quantità 0)
  #      omettere la riga contrassegnata.
  if not inventory:
    print("Nessun prodotto nel magazzino.")
    return
  print("PRODOTTO\tQUANTITA'\tPREZZO")
  for product in inventory:
    if product['quantità'] > 0:  #  <<<<<---------------filtro-----------------------
      # Mostra il prezzo di vendita mantenedo una larghezza fissa per allineare i print alle colonne
      print(f"{product['nome']:<20}{product['quantità']:<12}€{product['prezzo_vendita']:<8}")

#-------------------------------------------------------------------------------

def register_sale(inventory, sales):
  """
  Questa funzione registra una vendita con uno o più prodotti.
  Verifica che i prodotti esistano e se la quantità è sufficiente per l'operazione.
  Se la vendita è valida allora sottrae le quantità dal magazzino
  e ne registra l'operazione di vendita andata a buon fine.
  """
  sale_products=[]
  while True:
    name=input("Nome del prodotto: ").strip()
    try:
      quantity=int(input("Quantità: "))
    except ValueError:
      print("Errore: la quantità deve essere un numero intero.")
      continue

    # Controllo che la quantità non sia negativa
    if quantity < 0:
      print("Errore: la quantità non può essere negativa.")
      continue

    # Gestione dell'esistenza prodotto e sufficienza quantità
    product=search_product_by_name(inventory, name)
    if not product:
      print(f"Errore: il prodotto #{name} non esiste nel magazzino.")
      return  # Annulla l'intera operazione di vendita
    if product['quantità'] < quantity:
      print(f"Errore: quantità insufficiente per il prodotto #{name}.")
      return  # l'intera operazione di vendita

    # Aggiunta del prodotto venduto all'elenco
    sale_products.append({
      "nome": product['nome'],
      "quantità": quantity,
      "prezzo_vendita": product['prezzo_vendita'],
      "prezzo_acquisto": product['prezzo_acquisto']
    })

    another=input("Aggiungere un altro prodotto ? (si/no): ").strip().lower()
    if another!="si":
      break

  # Se tutti i prodotti sono disponibili esecuzione della vendita aggiornando il magazzino
  total=0.0
  for item in sale_products:
    # Aggiorna la quantità in inventario
    product=search_product_by_name(inventory, item['nome'])
    product['quantità']-=item['quantità']
    total+=item['quantità'] * item['prezzo_vendita']

  sale_record={"items": sale_products, "totale": total}
  sales.append(sale_record)
  save_inventory(inventory)
  save_sales(sales)

  # Stampa il riepilogo della vendita
  print("VENDITA REGISTRATA")
  for item in sale_products:
    print(f" - {item['quantità']} X {item['nome']}: €{item['prezzo_vendita']}")
  # Formattazione del totale con due cifre decimali
  print(f"\nTotale: €{total:.2f}")

#-------------------------------------------------------------------------------

def show_profits(sales):
  """
  Questa funzione calcola e mostra il profitto lordo ed il profitto netto.
  Profitto lordo = somma totale delle vendite;
  Profitto netto = profitto lordo - costo di acquisto dei prodotti venduti.
  """
  g_profit=0.0
  t_costs=0.0
  for sale in sales:
    g_profit+=sale['totale']
    for item in sale['items']:
      t_costs+=item['quantità'] * item['prezzo_acquisto']
  n_profit=g_profit - t_costs
  print(f"Profitto: lordo=€{g_profit:.2f} netto=€{n_profit:.2f}")

#-------------------------------------------------------------------------------

def show_help():
  """
  Questa funzione mostra i comandi disponibili.
  """
  help_message=(
    "I comandi disponibili sono i seguenti:\n"
    "--> aggiungi: aggiungi un prodotto al magazzino\n"
    "--> elenca: elenca i prodotto in magazzino\n"
    "--> vendita: registra una vendita effettuata\n"
    "--> profitti: mostra i profitti totali\n"
    "--> aiuto: mostra i possibili comandi\n"
    "--> chiudi: esci dal programma"
  )
  print(help_message)

#-------------------------------------------------------------------------------

def invalid_command():
  """
  Questa funzione gestisce il caso in cui il comando inserito non sia valido,
  mostrando un messaggio e il menu di comandi disponibili.
  """
  print("Comando non valido")
  print("I comandi disponibili sono i seguenti:")
  print("--> aggiungi: aggiungi un prodotto al magazzino")
  print("--> elenca: elenca i prodotto in magazzino")
  print("--> vendita: registra una vendita effettuata")
  print("--> profitti: mostra i profitti totali")
  print("--> aiuto: mostra i possibili comandi")
  print("--> chiudi: esci dal programma")

#-------------------------------------------------------------------------------

##########################     LOOP PRINCIPALE     ###########################
inventory=load_inventory()
sales=load_sales()

cmd=None
while cmd!="chiudi":
  cmd=input("\nInserisci un comando: ").strip().lower()
  if cmd=="vendita":
    register_sale(inventory, sales)
  elif cmd=="profitti":
    show_profits(sales)
  elif cmd=="aggiungi":
    add_product(inventory)
  elif cmd=="elenca":
    list_products(inventory)
  elif cmd=="aiuto":
    show_help()
  elif cmd=="chiudi":
    print("Bye bye")
  else:
    invalid_command()


Inserisci un comando: aiuto
I comandi disponibili sono i seguenti:
--> aggiungi: aggiungi un prodotto al magazzino
--> elenca: elenca i prodotto in magazzino
--> vendita: registra una vendita effettuata
--> profitti: mostra i profitti totali
--> aiuto: mostra i possibili comandi
--> chiudi: esci dal programma

Inserisci un comando: aggiungi
Nome del prodotto: latte di soia
Quantità: 20
Prezzo di acquisto: 0.8
Prezzo di vendita: 1.4
AGGIUNTO: 20 X latte di soia

Inserisci un comando: aggiungi
Nome del prodotto: tofu
Quantità: 10
Prezzo di acquisto: 2.2
Prezzo di vendita: 4.19
AGGIUNTO: 10 X tofu

Inserisci un comando: aggiungi
Nome del prodotto: seitan
Quantità: 5
Prezzo di acquisto: 3
Prezzo di vendita: 5.49
AGGIUNTO: 5 X seitan

Inserisci un comando: elenca
PRODOTTO	QUANTITA'	PREZZO
latte di soia       20          €1.4     
tofu                10          €4.19    
seitan              5           €5.49    

Inserisci un comando: vendita
Nome del prodotto: latte di soia
Quantità: 5
Ag